# Proyecto 1: Introducción a la IA Geoespacial (GeoAI)
## Parte 2 — Clasificación de Cobertura del Suelo (Land Cover Classification)

**Dataset:** MODIS Lake Powell (NASA CISTO) — HuggingFace  
**Objetivo:** Clasificar el tipo de cobertura del suelo a partir de bandas espectrales de MODIS  
**Modelo:** Gradient Boosting (XGBoost / scikit-learn)

---

## 1. Instalación e Importación de Librerías

In [ ]:
!pip install datasets shap -q
print('✅ Paquetes instalados')

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import datasets
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import shap

from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_theme(style='whitegrid', palette='muted')
print('✅ Librerías importadas correctamente')

## 2. Variables Generales

In [ ]:
DATASET_URL  = 'nasa-cisto-data-science-group/modis-lake-powell-toy-dataset'
RANDOM_STATE = 42

# Bandas espectrales MODIS + índices derivados
BAND_NAMES = [
    'sur_refl_b01_1',  # Banda 1 - Rojo
    'sur_refl_b02_1',  # Banda 2 - NIR
    'sur_refl_b03_1',  # Banda 3 - Azul
    'sur_refl_b04_1',  # Banda 4 - Verde
    'sur_refl_b05_1',  # Banda 5 - SWIR1
    'sur_refl_b06_1',  # Banda 6 - SWIR2
    'sur_refl_b07_1',  # Banda 7 - SWIR3
    'ndvi', 'ndwi1', 'ndwi2'
]

# Columnas de metadatos que no se usan como features
COLS_TO_DROP = ['x_offset', 'y_offset', 'year', 'julian_day', 'water']

os.makedirs('output', exist_ok=True)
print('✅ Variables configuradas')

## 3. Carga del Dataset

In [ ]:
%%time
dataset  = datasets.load_dataset(DATASET_URL, split='train')
df       = pd.DataFrame(dataset)
print(f'✅ Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas')
df.head()

In [ ]:
# Explorar todas las columnas disponibles para identificar la etiqueta de cobertura del suelo
print('--- Todas las columnas del dataset ---')
for col in df.columns:
    n_unique = df[col].nunique()
    dtype    = df[col].dtype
    print(f'  {col:<25} dtype={str(dtype):<10} unique={n_unique}')

In [ ]:
# El dataset solo tiene 'water'. Creamos clases de land cover
# a partir de umbrales espectrales — técnica estándar en teledetección MODIS
# Clases basadas en NDVI, NDWI y la etiqueta de agua existente

def asignar_land_cover(row):
    if row['water'] == 1:
        return 0   # Agua
    elif row['ndvi'] > 0.4:
        return 1   # Vegetación densa
    elif row['ndvi'] > 0.15:
        return 2   # Vegetación dispersa / Matorral
    elif row['ndwi1'] > 0.0:
        return 3   # Zona húmeda / Suelo mojado
    else:
        return 4   # Suelo desnudo / Roca / Arena

df['land_cover'] = df.apply(asignar_land_cover, axis=1)
LABEL_NAME = 'land_cover'

LC_NAMES = {
    0: 'Agua',
    1: 'Vegetación Densa',
    2: 'Vegetación Dispersa',
    3: 'Zona Húmeda',
    4: 'Suelo Desnudo'
}

print(f'✅ Columna "{LABEL_NAME}" creada con {df[LABEL_NAME].nunique()} clases')
print()
for cls, nombre in LC_NAMES.items():
    count = (df[LABEL_NAME] == cls).sum()
    pct = count / len(df) * 100
    print(f'  Clase {cls} — {nombre:<22} : {count:>7,} píxeles ({pct:.1f}%)')

In [ ]:
# Verificar distribución y estadísticas
clases_presentes = sorted(df[LABEL_NAME].unique())
nombres_clases   = [LC_NAMES[c] for c in clases_presentes]

print('Clases presentes:', clases_presentes)
print()
df[BAND_NAMES + [LABEL_NAME]].describe().round(4)

In [ ]:
# Estadísticas descriptivas
print(df[BAND_NAMES + [LABEL_NAME]].describe().round(4))
print(f'\nValores nulos por columna:\n{df.isnull().sum()[df.isnull().sum() > 0]}')

## 4. Análisis Exploratorio de Datos (EDA)
Realizamos al menos **4 visualizaciones** para entender la distribución de las coberturas del suelo.

In [ ]:
# Mapa de nombres MODIS Land Cover (MCD12Q1) para las clases más comunes
# Ajustar si el dataset usa etiquetas diferentes
LC_NAMES = {
    0:  'Agua',
    1:  'Bosque Perennifolio (Aguja)',
    2:  'Bosque Perennifolio (Hoja ancha)',
    3:  'Bosque Caducifolio (Aguja)',
    4:  'Bosque Caducifolio (Hoja ancha)',
    5:  'Bosque Mixto',
    6:  'Matorral Cerrado',
    7:  'Matorral Abierto',
    8:  'Sabana Arbolada',
    9:  'Sabana',
    10: 'Pastizal',
    11: 'Humedal Permanente',
    12: 'Tierra de Cultivo',
    13: 'Zona Urbana',
    14: 'Mosaico Cultivo/Vegetación Natural',
    15: 'Nieve y Hielo',
    16: 'Suelo Desnudo / Escasa Vegetación',
    17: 'Tundra'
}

# Aplicar nombres a las clases presentes en el dataset
clases_presentes = sorted(df[LABEL_NAME].unique())
nombres_clases   = [LC_NAMES.get(c, f'Clase {c}') for c in clases_presentes]
print('Clases en el dataset:')
for c, n in zip(clases_presentes, nombres_clases):
    count = (df[LABEL_NAME] == c).sum()
    print(f'  {c:>3} — {n:<45} ({count:,} píxeles)')

In [ ]:
# --- Visualización 1: Distribución de Clases de Cobertura del Suelo ---
counts = df[LABEL_NAME].value_counts().sort_index()
labels = [LC_NAMES.get(c, f'Clase {c}') for c in counts.index]
n_cls  = len(counts)
colors = plt.cm.tab20(np.linspace(0, 1, n_cls))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar chart
bars = axes[0].bar(range(n_cls), counts.values, color=colors, edgecolor='white', linewidth=1)
axes[0].set_xticks(range(n_cls))
axes[0].set_xticklabels([f'Clase {c}' for c in counts.index], rotation=45, ha='right')
axes[0].set_ylabel('Cantidad de Píxeles', fontsize=11)
axes[0].set_title('Distribución de Clases de Cobertura del Suelo', fontsize=12, fontweight='bold')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 f'{val:,}', ha='center', va='bottom', fontsize=8, rotation=0)

# Pie chart
wedges, texts, autotexts = axes[1].pie(
    counts.values, labels=[f'C{c}' for c in counts.index],
    autopct='%1.1f%%', colors=colors,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
for t in autotexts:
    t.set_fontsize(8)
axes[1].set_title('Proporción de Clases', fontsize=12, fontweight='bold')

# Leyenda
patches = [mpatches.Patch(color=colors[i], label=f'C{c}: {LC_NAMES.get(c, f"Clase {c}")}')
           for i, c in enumerate(counts.index)]
fig.legend(handles=patches, loc='lower center', ncol=3, fontsize=8,
           bbox_to_anchor=(0.5, -0.25), frameon=True)

plt.suptitle('Viz 1: Distribución de la Clase Objetivo — Land Cover', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('output/viz1_lc_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Visualización 2: Firma Espectral por Clase de Cobertura del Suelo ---
# Ordenar bandas por longitud de onda aproximada
spectral_bands  = ['sur_refl_b03_1','sur_refl_b04_1','sur_refl_b01_1',
                   'sur_refl_b02_1','sur_refl_b05_1','sur_refl_b06_1','sur_refl_b07_1']
wavelengths     = [470, 550, 650, 860, 1240, 1640, 2130]
band_short      = ['B3\n(Azul)','B4\n(Verde)','B1\n(Rojo)','B2\n(NIR)',
                   'B5\n(SWIR1)','B6\n(SWIR2)','B7\n(SWIR3)']

fig, ax = plt.subplots(figsize=(12, 6))

for i, cls in enumerate(clases_presentes):
    subset    = df[df[LABEL_NAME] == cls][spectral_bands]
    mean_vals = subset.mean().values
    ax.plot(wavelengths, mean_vals, marker='o', linewidth=2,
            color=colors[i], label=f'C{cls}: {LC_NAMES.get(cls, f"Clase {cls}")}')

ax.set_xticks(wavelengths)
ax.set_xticklabels([f'{w}nm\n{b}' for w, b in zip(wavelengths, band_short)], fontsize=9)
ax.set_xlabel('Longitud de Onda / Banda', fontsize=11)
ax.set_ylabel('Reflectancia Superficial Media', fontsize=11)
ax.set_title('Viz 2: Firmas Espectrales por Clase de Cobertura del Suelo\n(Instrumento: MODIS)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=8, loc='upper right', ncol=2, framealpha=0.9)
ax.axvline(700, color='gray', linestyle=':', alpha=0.5, label='Límite Visible/NIR')
ax.axvspan(400, 700, alpha=0.04, color='yellow')
ax.axvspan(700, 2200, alpha=0.04, color='red')
plt.tight_layout()
plt.savefig('output/viz2_spectral_signatures.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Visualización 3: Heatmap NDVI / NDWI por clase de cobertura ---
indices_cols = ['ndvi', 'ndwi1', 'ndwi2']

# Calcular medias de los índices por clase
pivot = df.groupby(LABEL_NAME)[indices_cols].mean()
pivot.index = [f'C{c}: {LC_NAMES.get(c, f"Clase {c}")[:20]}' for c in pivot.index]

fig, ax = plt.subplots(figsize=(8, max(4, len(pivot)*0.5 + 1)))
sns.heatmap(
    pivot, annot=True, fmt='.3f', cmap='RdYlGn',
    linewidths=0.5, ax=ax,
    cbar_kws={'label': 'Valor del Índice'},
    xticklabels=['NDVI\n(Vegetación)', 'NDWI1\n(Agua 1)', 'NDWI2\n(Agua 2)']
)
ax.set_title('Viz 3: Media de Índices Espectrales por Clase de Cobertura del Suelo',
             fontsize=12, fontweight='bold')
ax.set_ylabel('Clase de Cobertura')
ax.set_xlabel('')
plt.tight_layout()
plt.savefig('output/viz3_indices_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Visualización 4: Scatter NIR vs Rojo coloreado por clase ---
# NIR vs Rojo es uno de los espacios más discriminativos en teledetección
sample = df.sample(n=min(6000, len(df)), random_state=RANDOM_STATE)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# NIR vs Rojo
for i, cls in enumerate(clases_presentes):
    sub = sample[sample[LABEL_NAME] == cls]
    axes[0].scatter(sub['sur_refl_b01_1'], sub['sur_refl_b02_1'],
                    alpha=0.35, s=8, color=colors[i],
                    label=f'C{cls}: {LC_NAMES.get(cls, f"Clase {cls}")[:15]}')
axes[0].set_xlabel('Banda 1 — Rojo (~650nm)', fontsize=11)
axes[0].set_ylabel('Banda 2 — NIR (~860nm)', fontsize=11)
axes[0].set_title('NIR vs Rojo por Clase', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=7, markerscale=2)

# NDVI vs NDWI1
for i, cls in enumerate(clases_presentes):
    sub = sample[sample[LABEL_NAME] == cls]
    axes[1].scatter(sub['ndwi1'], sub['ndvi'],
                    alpha=0.35, s=8, color=colors[i],
                    label=f'C{cls}')
axes[1].axhline(0, color='gray', linestyle='--', lw=1, alpha=0.6)
axes[1].axvline(0, color='gray', linestyle='--', lw=1, alpha=0.6)
axes[1].set_xlabel('NDWI1', fontsize=11)
axes[1].set_ylabel('NDVI', fontsize=11)
axes[1].set_title('NDWI1 vs NDVI por Clase', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=7, markerscale=2)

plt.suptitle('Viz 4: Separabilidad Espectral entre Clases de Cobertura del Suelo',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('output/viz4_scatter_classes.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Ingeniería de Características (Feature Engineering)

In [ ]:
df_fe = df.copy()

# --- Feature 1: Enhanced Vegetation Index (EVI) ---
# Más robusto que el NDVI para vegetación densa y corrección atmosférica
G, C1, C2, L = 2.5, 6, 7.5, 1
nir  = df_fe['sur_refl_b02_1']
red  = df_fe['sur_refl_b01_1']
blue = df_fe['sur_refl_b03_1']
df_fe['evi'] = G * ((nir - red) / (nir + C1*red - C2*blue + L + 1e-6))

# --- Feature 2: Soil Adjusted Vegetation Index (SAVI) ---
# Reduce el efecto del suelo en áreas con vegetación dispersa
df_fe['savi'] = ((nir - red) / (nir + red + 0.5)) * 1.5

# --- Feature 3: Normalized Difference Built-up Index (NDBI) ---
# Detecta zonas urbanas y construidas
swir = df_fe['sur_refl_b06_1']
df_fe['ndbi'] = (swir - nir) / (swir + nir + 1e-6)

# --- Feature 4: Ratio NIR / SWIR (sensible a contenido de humedad) ---
df_fe['nir_swir_ratio'] = nir / (swir + 1e-6)

# --- Feature 5: Brightness (suma ponderada de bandas visibles) ---
df_fe['brightness'] = (red + df_fe['sur_refl_b04_1'] + blue) / 3

new_features = ['evi', 'savi', 'ndbi', 'nir_swir_ratio', 'brightness']
print(f'✅ {len(new_features)} features creadas:')
for f in new_features:
    print(f'   - {f}')

df_fe[new_features].describe().round(4)

In [ ]:
# Columnas finales de features y etiqueta
FEATURE_COLS = BAND_NAMES + new_features

X = df_fe[FEATURE_COLS].values
y_raw = df_fe[LABEL_NAME].values

# Codificar la etiqueta (necesario para algunos modelos)
le = LabelEncoder()
y  = le.fit_transform(y_raw)

class_names = [LC_NAMES.get(c, f'Clase {c}') for c in le.classes_]
print(f'✅ X shape : {X.shape}')
print(f'   y shape : {y.shape}')
print(f'   Clases  : {list(le.classes_)} → {list(range(len(le.classes_)))}')

## 6. División del Dataset (Train / Test Split)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y   # Mantiene la proporción de todas las clases
)

print(f'✅ Entrenamiento : {X_train.shape[0]:,} muestras')
print(f'   Prueba        : {X_test.shape[0]:,} muestras')
print(f'   # Clases      : {len(np.unique(y))}')

## 7. Entrenamiento del Modelo

In [ ]:
# Gradient Boosting — excelente para clasificación multiclase con datos tabulares
# Construye árboles secuencialmente, cada uno corrigiendo los errores del anterior
model = GradientBoostingClassifier(
    n_estimators=200,       # Número de árboles (etapas de boosting)
    learning_rate=0.1,      # Peso de cada árbol en la corrección
    max_depth=5,            # Profundidad máxima de cada árbol
    min_samples_split=10,   # Mínimo de muestras para dividir un nodo
    subsample=0.8,          # Fracción de muestras por árbol (reduce overfitting)
    random_state=RANDOM_STATE
)

print('🌲 Entrenando Gradient Boosting Classifier...')
print('   (Esto puede tardar unos minutos con datasets grandes)')
%time model.fit(X_train, y_train)
print('✅ Modelo entrenado exitosamente')

## 8. Predicciones y Evaluación del Modelo

In [ ]:
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f'🎯 Exactitud (Accuracy): {acc:.4f} ({acc*100:.2f}%)')
print()
print('--- Reporte de Clasificación ---')
print(classification_report(y_test, y_pred, target_names=class_names))

In [ ]:
# --- Matriz de Confusión ---
cm   = confusion_matrix(y_test, y_pred)
n_c  = len(le.classes_)
fig_w = max(8, n_c * 1.2)

fig, ax = plt.subplots(figsize=(fig_w, fig_w * 0.8))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[f'C{c}' for c in le.classes_]
)
disp.plot(ax=ax, cmap='Blues', colorbar=True, xticks_rotation=45)
ax.set_title(f'Matriz de Confusión — Land Cover Classification\nAccuracy: {acc:.4f}',
             fontsize=13, fontweight='bold')

# Leyenda de clases abajo
legend_text = ' | '.join([f'C{c}: {LC_NAMES.get(c, f"Clase {c}")[:18]}' for c in le.classes_])
fig.text(0.5, -0.02, legend_text, ha='center', fontsize=7, wrap=True)

plt.tight_layout()
plt.savefig('output/eval_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Precisión por clase (F1-Score) ---
from sklearn.metrics import f1_score

f1_per_class = f1_score(y_test, y_pred, average=None)
f1_df = pd.DataFrame({
    'Clase': [f'C{c}: {LC_NAMES.get(c, f"Clase {c}")[:25]}' for c in le.classes_],
    'F1-Score': f1_per_class
}).sort_values('F1-Score', ascending=True)

fig, ax = plt.subplots(figsize=(10, max(4, len(f1_df) * 0.5)))
bar_colors = ['#E85D4C' if v < 0.7 else '#4C9BE8' if v < 0.9 else '#2ECC71'
              for v in f1_df['F1-Score']]
ax.barh(f1_df['Clase'], f1_df['F1-Score'], color=bar_colors, edgecolor='white')
ax.axvline(0.7, color='orange', linestyle='--', lw=1.5, label='Umbral 0.70')
ax.axvline(0.9, color='green',  linestyle='--', lw=1.5, label='Umbral 0.90')
for i, (_, row) in enumerate(f1_df.iterrows()):
    ax.text(row['F1-Score'] + 0.005, i, f"{row['F1-Score']:.3f}", va='center', fontsize=9)
ax.set_xlabel('F1-Score', fontsize=12)
ax.set_title('F1-Score por Clase de Cobertura del Suelo', fontsize=12, fontweight='bold')
ax.set_xlim(0, 1.08)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('output/eval_f1_per_class.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. IA Explicable (XAI) — Importancia de Características

Dos enfoques:
1. **Feature Importance intrínseca** del Gradient Boosting
2. **SHAP values** — explicaciones locales y globales

In [ ]:
# --- XAI 1: Feature Importance del Gradient Boosting ---
importances = model.feature_importances_
fi_df = pd.DataFrame({
    'Feature': FEATURE_COLS,
    'Importance': importances
}).sort_values('Importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
palette = ['#2ECC71' if f in new_features else '#4C9BE8' for f in fi_df['Feature']]
ax.barh(fi_df['Feature'][::-1], fi_df['Importance'][::-1],
        color=palette[::-1], edgecolor='white')

for i, (_, row) in enumerate(fi_df[::-1].iterrows()):
    ax.text(row['Importance'] + 0.001, i, f"{row['Importance']:.4f}", va='center', fontsize=9)

# Leyenda de colores
p_orig = mpatches.Patch(color='#4C9BE8', label='Bandas originales MODIS')
p_new  = mpatches.Patch(color='#2ECC71', label='Features de ingeniería')
ax.legend(handles=[p_orig, p_new], fontsize=10)

ax.set_xlabel('Importancia (Gradient Boosting)', fontsize=12)
ax.set_title('XAI: Importancia de Características — Land Cover Classification',
             fontsize=12, fontweight='bold')
ax.set_xlim(0, fi_df['Importance'].max() * 1.15)
plt.tight_layout()
plt.savefig('output/xai1_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 5 características:')
print(fi_df.head().to_string(index=False))

In [ ]:
# PermutationExplainer — compatible con cualquier modelo sklearn multiclase
print('🔍 Calculando SHAP values con PermutationExplainer...')
print('   (Usamos muestra pequeña para mantener tiempo razonable)')

sample_size = min(150, len(X_test))
idx_sample  = np.random.RandomState(RANDOM_STATE).choice(len(X_test), sample_size, replace=False)
X_shap      = X_test[idx_sample]

# Fondo de referencia: muestra del training set
background  = shap.maskers.Independent(X_train[:100])

explainer   = shap.PermutationExplainer(model.predict_proba, background)
shap_exp    = explainer(X_shap)

# shap_exp.values shape: (n_samples, n_features, n_classes)
sv_all  = shap_exp.values                        # array 3D completo
sv_mean = np.abs(sv_all).mean(axis=(0, 2))       # importancia global: promedio sobre muestras y clases
sv_cls0 = sv_all[:, :, 0]                        # SHAP para clase 0 (para beeswarm)

print(f'✅ SHAP values calculados')
print(f'   shap_exp.values shape : {sv_all.shape}  → (muestras, features, clases)')

In [ ]:
# --- SHAP: Bar Plot — Importancia global promedio (todas las clases) ---
mean_abs_shap = sv_mean.mean(axis=0)
shap_fi_df = pd.DataFrame({
    'Feature': FEATURE_COLS,
    'SHAP_Importance': mean_abs_shap
}).sort_values('SHAP_Importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
bar_colors = ['#2ECC71' if f in new_features else '#4C9BE8' for f in shap_fi_df['Feature']]
ax.barh(shap_fi_df['Feature'], shap_fi_df['SHAP_Importance'],
        color=bar_colors, edgecolor='white')

for i, (_, row) in enumerate(shap_fi_df.iterrows()):
    ax.text(row['SHAP_Importance'] + 0.0001, i,
            f"{row['SHAP_Importance']:.4f}", va='center', fontsize=9)

ax.legend(handles=[p_orig, p_new], fontsize=10)
ax.set_xlabel('|SHAP| Promedio (todas las clases)', fontsize=12)
ax.set_title('XAI: SHAP — Importancia Global de Características\n(Land Cover Classification)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('output/xai2_shap_global.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- SHAP: Beeswarm para la primera clase (muestra dirección del impacto) ---
plt.figure(figsize=(10, 7))
shap.summary_plot(
    sv_cls0,
    X_shap,
    feature_names=FEATURE_COLS,
    show=False,
    plot_type='dot',
    title=f'SHAP Beeswarm — Clase: {class_names[0]}'
)
plt.title(f'XAI: SHAP Beeswarm — Clase "{class_names[0]}"\n'
          f'(Rojo = valor alto de feature, impacta positivamente la predicción)',
          fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('output/xai3_shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Resumen Final ---
print('=' * 58)
print('  RESUMEN — Clasificación de Cobertura del Suelo (MODIS)')
print('=' * 58)
print(f'  Modelo       : Gradient Boosting (n=200 árboles)')
print(f'  Features     : {len(FEATURE_COLS)} (10 originales + 5 ingeniería)')
print(f'  # Clases     : {len(le.classes_)}')
print(f'  Train size   : {X_train.shape[0]:,} muestras')
print(f'  Test size    : {X_test.shape[0]:,} muestras')
print(f'  Accuracy     : {acc*100:.2f}%')
print(f'  Top feature (GB)   : {fi_df.iloc[0]["Feature"]} ({fi_df.iloc[0]["Importance"]:.4f})')
print(f'  Top feature (SHAP) : {shap_fi_df.iloc[-1]["Feature"]} ({shap_fi_df.iloc[-1]["SHAP_Importance"]:.4f})')
print('=' * 58)
print('✅ Parte 2 completada. Archivos guardados en /output/')